# Batch Normalization: Forward Pass & Running Stats

Companion notebook for the wiki page: [Batch Normalization: Forward Pass & Running Stats](https://ml-viz-ruby.vercel.app/wiki/batchnorm-algorithm)

We implement a BatchNorm layer from scratch, verify the worked example, simulate running statistics, and visualise how BatchNorm shifts activation distributions through a deep network.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use('dark_background')
rng = np.random.default_rng(42)

## From-scratch BatchNorm layer

In [ ]:
class BatchNorm:
    """
    BatchNorm layer with forward pass, running statistics, and backward pass.
    x: (N, D) — N samples, D features
    """
    def __init__(self, D, eps=1e-5, momentum=0.1):
        self.gamma = np.ones(D)
        self.beta  = np.zeros(D)
        self.eps   = eps
        self.alpha = momentum       # EMA momentum
        self.mu_run  = np.zeros(D)
        self.var_run = np.ones(D)
        self.cache   = None

    def forward(self, x, training=True):
        if training:
            mu  = x.mean(axis=0)             # (D,)
            var = x.var(axis=0)              # (D,)
            x_hat = (x - mu) / np.sqrt(var + self.eps)
            # Update running statistics
            self.mu_run  = (1 - self.alpha) * self.mu_run  + self.alpha * mu
            self.var_run = (1 - self.alpha) * self.var_run + self.alpha * var
            self.cache = (x, x_hat, mu, var)
        else:
            # Inference: use running statistics
            x_hat = (x - self.mu_run) / np.sqrt(self.var_run + self.eps)
        return self.gamma * x_hat + self.beta

    def backward(self, dy):
        """Backward pass — gradients wrt x, gamma, beta."""
        x, x_hat, mu, var = self.cache
        N, D = x.shape
        dgamma = (dy * x_hat).sum(axis=0)
        dbeta  = dy.sum(axis=0)
        dx_hat = dy * self.gamma
        dvar = (dx_hat * (x - mu) * -0.5 * (var + self.eps)**(-1.5)).sum(axis=0)
        dmu  = (-dx_hat / np.sqrt(var + self.eps)).sum(axis=0) + \
               dvar * (-2 * (x - mu)).mean(axis=0)
        dx   = dx_hat / np.sqrt(var + self.eps) + \
               dvar * 2 * (x - mu) / N + dmu / N
        return dx, dgamma, dbeta

## Reproduce the 4-element worked example

In [ ]:
# x = [2, 4, 6, 8], γ = 2, β = 1
x_ex = np.array([[2.0], [4.0], [6.0], [8.0]])  # (4, 1)
bn_ex = BatchNorm(D=1, eps=0)
bn_ex.gamma = np.array([2.0])
bn_ex.beta  = np.array([1.0])

y_ex = bn_ex.forward(x_ex, training=True)

print("Step 1 — statistics:")
print(f"  μ_B = {x_ex.mean():.4f}  (expected 5)")
print(f"  σ²_B = {x_ex.var():.4f}  (expected 5)")

x_hat_ex = (x_ex - x_ex.mean()) / x_ex.std()
print("\nStep 2 — normalized:")
print(f"  x̂ = {x_hat_ex.ravel().round(3)}  (expected ≈ [-1.342, -0.447, 0.447, 1.342])")

print("\nStep 3 — rescaled (γ=2, β=1):")
print(f"  y = {y_ex.ravel().round(3)}  (expected ≈ [-1.683, 0.106, 1.894, 3.683])")

print(f"\nVerify: mean(y) = {y_ex.mean():.3f} = β = 1  ✓" if abs(y_ex.mean()-1)<1e-9 else "FAIL")
print(f"Verify: var(y)  = {y_ex.var():.3f} = γ² = 4  ✓" if abs(y_ex.var()-4)<1e-6 else "FAIL")

## Running statistics simulation

In [ ]:
# Simulate 30 mini-batches arriving during training
mu_run, var_run = 0.0, 1.0
alpha = 0.1

# True data distribution: N(5, 5)
batch_mus  = rng.normal(5.0, 0.5, 30)
batch_vars = rng.normal(5.0, 0.5, 30)

mu_history  = [mu_run]
var_history = [var_run]

for mu_b, var_b in zip(batch_mus, batch_vars):
    mu_run  = (1 - alpha) * mu_run  + alpha * mu_b
    var_run = (1 - alpha) * var_run + alpha * var_b
    mu_history.append(mu_run)
    var_history.append(var_run)

print(f"Running mean after 30 batches: {mu_run:.4f}  (should converge ≈ 5)")
print(f"Running var  after 30 batches: {var_run:.4f}  (should converge ≈ 5)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

for ax, hist, true_val, label, color in [
    (ax1, mu_history,  5.0, 'Running mean μ_run', '#6366f1'),
    (ax2, var_history, 5.0, 'Running var σ²_run', '#22d3ee'),
]:
    ax.plot(hist, color=color, lw=2, label=label)
    ax.axhline(true_val, color='#f59e0b', ls='--', label=f'True value = {true_val}')
    ax.set_xlabel('Batch'); ax.set_ylabel('Value')
    ax.set_title(label); ax.legend(); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.suptitle('EMA running statistics converging to true distribution', y=1.02)
plt.show()

## Activation distributions: before vs after BatchNorm

In [ ]:
# Simulate pre-activations for 5 layers without BN (distributions drift)
n_layers = 5
n_samples = 512

# Without BatchNorm: each layer multiplies by a random weight matrix
x = rng.normal(0, 1, (n_samples, 64))
activations_no_bn  = [x]
activations_with_bn = [x]

for layer in range(n_layers):
    W = rng.normal(0, 1.5, (64, 64))  # large weights → drift
    # Without BN
    x_no = np.maximum(0, activations_no_bn[-1] @ W)   # ReLU
    activations_no_bn.append(x_no)
    # With BN
    bn = BatchNorm(64)
    pre_act = activations_with_bn[-1] @ W
    x_bn = np.maximum(0, bn.forward(pre_act))          # BN then ReLU
    activations_with_bn.append(x_bn)

fig, axes = plt.subplots(2, n_layers + 1, figsize=(14, 5))

for i, (acts_no, acts_bn) in enumerate(
    zip(activations_no_bn, activations_with_bn)
):
    flat_no = acts_no[:, :4].ravel()
    flat_bn = acts_bn[:, :4].ravel()
    # Remove inf/nan
    flat_no = flat_no[np.isfinite(flat_no)]
    flat_bn = flat_bn[np.isfinite(flat_bn)]
    if len(flat_no):
        lim = np.percentile(np.abs(flat_no), 99) if len(flat_no) else 5
        axes[0][i].hist(flat_no, bins=40, color='#ef4444', alpha=0.8,
                        range=(-lim, lim))
    if len(flat_bn):
        axes[1][i].hist(flat_bn, bins=40, color='#6366f1', alpha=0.8,
                        range=(-3, 3) if i > 0 else (-4, 4))
    title = 'Input' if i == 0 else f'After layer {i}'
    axes[0][i].set_title(title, fontsize=9)
    axes[1][i].set_title(title, fontsize=9)
    axes[0][i].set_yticks([]); axes[1][i].set_yticks([])

axes[0][0].set_ylabel('Without BN', color='#ef4444', fontsize=10)
axes[1][0].set_ylabel('With BN', color='#6366f1', fontsize=10)
plt.suptitle('Activation distributions across layers — BN prevents drift',
             y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## ✏️ Your turn

**Exercise 1:** Implement `LayerNorm` — same formula but normalize over the **feature dimension** (axis=-1) instead of the batch dimension (axis=0). Verify it works at batch size 1.

**Exercise 2:** Show that BatchNorm is **permutation equivariant** in the batch dimension: shuffling the rows of `x` before `forward()` produces the same output (in shuffled order). Write a numerical test.

**Exercise 3:** Implement the `backward` pass independently (without looking at the class above) and verify it against finite differences.

In [ ]:
# Exercise 1: LayerNorm
class LayerNorm:
    def __init__(self, D, eps=1e-5):
        self.gamma = np.ones(D)
        self.beta  = np.zeros(D)
        self.eps   = eps

    def forward(self, x):
        # TODO(you): normalize over axis=-1 (feature dimension)
        pass

# Test: should work even with N=1
# ln = LayerNorm(4)
# x_single = np.array([[2.0, 4.0, 6.0, 8.0]])
# y_ln = ln.forward(x_single)
# assert abs(y_ln.mean()) < 1e-10, "LayerNorm should give zero mean per sample"

<details>
<summary>Solution — Exercise 1</summary>

```python
def forward(self, x):
    mu  = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    x_hat = (x - mu) / np.sqrt(var + self.eps)
    return self.gamma * x_hat + self.beta
```
</details>